# Chronos follow-up across four periods

Eight series, four datasets, four periods ending in March, June, September and
December of each dataset's selected year. Each period has 42-day validation and
test calendar blocks, with a 168-hour validation embargo. Complete test forecasts
must finish by the period endpoint, giving up to 840 hourly origins per window.
The French household is excluded in Q1 and Q2: gaps prevent complete validation/test
setups under the original missing-data rule. It remains included in Q3 and Q4.
Quarter labels are calendar
labels, not identical seasons across countries.

Daily/weekly/blend/hybrid definitions, 336-hour context, 0.5 quantile and model ID
are unchanged. Blend weights and baseline choice are selected afresh using only
the earlier validation period for each window. No tuning uses that window's test
answers. The model commit is fixed to the completed pilot's commit.

This is a follow-up designed after inspecting the original pilot. December
repeats the original evaluation period, so do not count it as new evidence.
Earlier windows cover more seasonal conditions, not additional years.
Pretraining overlap is not excluded. Forecasts overlap within a window, so
uncertainty analysis must still account for time dependence.

**Run:** choose Runtime > Change runtime type > T4 GPU. Run cells in order.
Upload `expanded_chronos_inputs.zip` when prompted. No GitHub push is needed.
Forecasting packages are installed in an isolated environment so Colab's Gradio
and Diffusers dependencies cannot conflict. Initial setup downloads PyTorch and
can take several minutes. The notebook's own environment is left unchanged.

In [ ]:
import os, subprocess, sys
from pathlib import Path

ENV = Path("/content/chronos_forecast_env")
PYTHON = str(ENV / "bin/python")
subprocess.run([sys.executable, "-m", "venv", str(ENV)], check=True)
# Do not inherit external Python package paths into the isolated process.
RUN_ENV = {key: value for key, value in os.environ.items()
           if key not in ("PYTHONPATH", "PYTHONHOME")}
RUN_ENV["PYTHONNOUSERSITE"] = "1"
subprocess.run([PYTHON, "-m", "pip", "install", "--quiet",
                "torch==2.11.0", "chronos-forecasting==2.2.2",
                "transformers==4.57.6", "numpy==2.1.3", "pandas==2.2.3"],
               env=RUN_ENV, check=True)
subprocess.run([PYTHON, "-m", "pip", "check"], env=RUN_ENV, check=True)

In [ ]:
subprocess.run([PYTHON, "-c", "import torch; from chronos import BaseChronosPipeline; "
                "assert torch.cuda.is_available(), 'Select a GPU runtime first'; "
                "print(torch.__version__, torch.cuda.get_device_name(0))"],
               env=RUN_ENV, check=True)

In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, zipfile, subprocess, sys, json

ROOT = Path("/content/expanded_chronos")
ROOT.mkdir(exist_ok=True)
uploaded = files.upload()
assert len(uploaded) == 1, "Choose only expanded_chronos_inputs.zip"
filename, payload = next(iter(uploaded.items()))
assert hashlib.sha256(payload).hexdigest() == "a64b6240507cb5e4ac8349e8265ab87e50c598d225400f15bbbc4e298cd74b58", "Wrong or changed ZIP"
with zipfile.ZipFile(filename) as archive:
    for member in archive.infolist():
        parts = PurePosixPath(member.filename)
        assert not parts.is_absolute() and ".." not in parts.parts and "\\" not in member.filename
        assert (ROOT / member.filename).resolve().is_relative_to(ROOT.resolve())
    archive.extractall(ROOT)
del uploaded, payload
for label in ("Q1", "Q2", "Q3", "Q4"):
    base = ROOT / label
    subprocess.run([PYTHON, str(base / "src/chronos_pilot.py"),
                    "--root", str(base), "--check-only"], env=RUN_ENV, check=True)

## Run all four periods

60 series/horizon/window cases in total. The model is loaded once per window.
Completed cases are saved under `results/Q1` etc. Rerunning this cell reuses
completed cases with an identical configuration. Runtime deletion removes local
progress, so download partial results before disconnecting if necessary.

The 168-hour task still uses Bolt's built-in extension beyond its native 64 steps.
No observed future values enter the model. Float32 and batch size 8 support T4.
December is rerun to keep outputs self-contained and comparable; the earlier
December ZIP remains preserved in the local project.

In [ ]:
OUTPUT = ROOT / "results"
for label in ("Q1", "Q2", "Q3", "Q4"):
    base = ROOT / label
    subprocess.run([PYTHON, "-u", str(base / "src/chronos_pilot.py"),
                    "--root", str(base), "--output", str(OUTPUT / label),
                    "--batch-size", "8", "--model-revision", "5d9f166d69f47aef3401367a7b842e78fe97b121"], env=RUN_ENV, check=True)

In [ ]:
import csv
rows = []
for label in ("Q1", "Q2", "Q3", "Q4"):
    assert (OUTPUT / label / "complete.json").exists(), f"Incomplete window: {label}"
    with (OUTPUT / label / "comparison.csv").open(newline="") as stream:
        rows.extend({"window": label, **row} for row in csv.DictReader(stream))
with (OUTPUT / "all_windows_comparison.csv").open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
for row in rows:
    if row["baseline"] == "selected" and row["horizon_hours"] == "168":
        print(row["window"], row["dataset"], row["series"], row["baseline_rule"],
              "Chronos gain:", round(float(row["chronos_MAE_gain_pct"]), 2), "%")
print("Positive means lower Chronos MAE. These are descriptive comparisons, not significance tests.")

In [ ]:
import shutil
assert OUTPUT.exists(), "No results yet"
result_zip = shutil.make_archive("/content/expanded_chronos_results", "zip", OUTPUT)
files.download(result_zip)